## 1 setup

In [6]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy scipy scikit-learn tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import ast
import json
import math
import re #espressioni regolari
from pathlib import Path

import joblib #salvare oggetti python
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix,save_npz,load_npz 
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer #trasforma testi dei prodotti in vettori numerici
from sklearn.preprocessing import normalize 
from tqdm.auto import tqdm

pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",160)
pd.set_option("display.float_format",lambda x:f"{x:,.6f}")

In [8]:
CATEGORY="Electronics"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg"
PRIMARY_K=10

MAX_EVAL_ROWS=None
SAVE_RANKS=True
SAVE_CONTENT_ARTIFACTS=True

MAX_TFIDF_FEATURES=20000 #20000 termini/caratteristiche testuali al max
TFIDF_MIN_DF=2 #tiene parole che compaiono in almeno due prodotti
TFIDF_MAX_DF=0.85 #elimina parole che compaiono nell 85% dei prodotti
TFIDF_NGRAM_RANGE=(1,2)  #usa sia parole singole, sia doppie
TFIDF_STOP_WORDS="english"  #rimuove parole comuni inglesi

TRAIN_SVD_FALLBACK=True 
SVD_FALLBACK_FACTORS=4
SVD_N_ITER=7

DEMO_USERS_N=3
DEMO_TOP_N=10
DEMO_SIMILAR_ITEMS_N=10

HYBRID_CONFIGS={ #combinare content-based, svd e popularity
    "Hybrid_Content70_Pop30":{"content":0.70,"svd":0.00,"popularity":0.30},
    "Hybrid_SVD70_Content30":{"content":0.30,"svd":0.70,"popularity":0.00},
    "Hybrid_SVD50_Content25_Pop25":{"content":0.25,"svd":0.50,"popularity":0.25},
    "Hybrid_SVD40_Content40_Pop20":{"content":0.40,"svd":0.40,"popularity":0.20},
}

## 2 caricamento dati

In [9]:
PROJECT_ROOT=Path("..").resolve() if (Path("..")/"data").exists() else Path(".").resolve()
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK05_DIR=RECOMMENDER_DIR/"notebook05_content_hybrid"
NOTEBOOK05_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"
QA_PATH=RECOMMENDER_DIR/"qa.parquet"

NOTEBOOK04_METRICS_PATH=NOTEBOOK04_DIR/"notebook04_metrics.csv"
NOTEBOOK04_COMPARISON_PATH=NOTEBOOK04_DIR/"notebook04_comparison.csv"
NOTEBOOK04_SUMMARY_PATH=NOTEBOOK04_DIR/"notebook04_summary.json"
POPULARITY_SCORES_PATH=NOTEBOOK04_DIR/"popularity_scores.parquet"
SVD_USER_EMB_PATH=NOTEBOOK04_DIR/"best_svd_user_embeddings.npy"
SVD_ITEM_EMB_PATH=NOTEBOOK04_DIR/"best_svd_item_embeddings.npy"
SVD_INFO_PATH=NOTEBOOK04_DIR/"best_svd_info.json"

required_paths={
    "train":TRAIN_PATH,
    "validation_eval":VALIDATION_EVAL_PATH,
    "test_eval":TEST_EVAL_PATH,
    "negative_samples_validation":VALIDATION_NEGATIVE_PATH,
    "negative_samples_test":TEST_NEGATIVE_PATH,
    "items":ITEMS_PATH,
    "user_mapping":USER_MAPPING_PATH,
    "item_mapping":ITEM_MAPPING_PATH,
}

missing_paths=[name for name,path in required_paths.items() if not path.exists()]

print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)
print("Notebook05 dir:",NOTEBOOK05_DIR)

for name,path in required_paths.items():
    print(f"{name}:{path} | exists={path.exists()}")

if missing_paths:
    raise FileNotFoundError("File mancanti dal Notebook 03:"+",".join(missing_paths))

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook04_models
Notebook05 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\notebook05_content_hybrid
train:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\train.parquet | exists=True
validation_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\validation_eval.parquet | exists=True
test_eval:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\test_eval.parquet | exists=True
negative_samples_validation:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_samples_validation.parquet | exists=True
negative_samples_test:C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender\negative_

In [10]:
#file parquet principali
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)

if SPLIT_SUMMARY_PATH.exists():
    split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))
else:
    split_summary={}

if NOTEBOOK04_SUMMARY_PATH.exists():
    notebook04_summary=json.loads(NOTEBOOK04_SUMMARY_PATH.read_text(encoding="utf-8"))
else:
    notebook04_summary={}

def parse_negative_items(value): #converte colonna negative_item_idxs in lista pulita di interi
    if value is None:
        return []
    if isinstance(value,np.ndarray):
        value=value.tolist()
    if isinstance(value,(list,tuple,set)):
        return [int(item) for item in value]
    if isinstance(value,str):
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()
validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items)
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None:
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

loaded_shapes=pd.DataFrame([
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))
display(pd.DataFrame.from_dict(notebook04_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,287752,14
1,validation_eval,131376,14
2,test_eval,131352,14
3,validation_samples,131376,7
4,test_samples,131352,7
5,items,12306,13
6,user_mapping,131398,2
7,item_mapping,12306,2


,value
category,Electronics
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
random_state,42
temporal_column,datetime
original_candidate_interactions,844060
positive_interactions_before_refilter,671903


,value
category,Electronics
primary_metric,ndcg
primary_k,10
best_model_validation,MostPopular
best_validation_score,0.284317
models_evaluated,"[MostPopular, ItemKNN_5, ItemKNN_10, ItemKNN_50, ItemKNN_100, SVD_2, SVD_4, SVD_6, SVD_8, SVD_10, SVD_12, SVD_16, SVD_24, SVD_32]"
train_rows,287752
validation_samples,131376
test_samples,131352
n_users,131398


## 3 costruzione del testo per ogni prodotto
metadati come nome, marca, descrizione, brand

In [11]:


if "item_idx" not in items_df.columns:
    items_df=items_df.merge(item_mapping_df,on="parent_asin",how="left")

items_df=items_df.copy()
items_df["item_idx"]=items_df["item_idx"].astype(np.int64)


def is_missing_value(value): #per vedere se un valore è mancante, None, NaN 
    if value is None:
        return True
    try:
        result=pd.isna(value)
        if isinstance(result,(bool,np.bool_)):
            return bool(result)
    except (TypeError,ValueError):
        return False
    return False


def parse_json_like(value): #converte un valore in una forma python
    if is_missing_value(value):
        return []
    if isinstance(value,(list,tuple,set,dict)):
        return value
    if isinstance(value,str): #se è stringa la pulisce dagli spazi
        text=value.strip()
        if not text or text.lower() in {"none","nan","null"}:
            return []
        try:
            return json.loads(text) #se è json valido
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(text) #se quello di prima non è andato bene
            except (SyntaxError,ValueError):
                return text #se niente va, restituisce testo originale
    return value


def flatten_text(value,max_parts=40): #prende valore complesso come lista o diz e lo trasforma in una singola stringa testuale
    parsed=parse_json_like(value)
    parts=[]

    def add_part(obj): #ricorsiva
        if len(parts)>=max_parts:
            return
        if is_missing_value(obj):
            return
        if isinstance(obj,dict):  #se è dizionario, scorre chiavi e valori
            for key,val in obj.items():
                if len(parts)>=max_parts:
                    break
                if is_missing_value(val):
                    continue
                if isinstance(val,(list,tuple,set,dict)):
                    add_part(val)
                else:
                    text=str(val).strip()
                    if text:
                        parts.append(str(key)+" "+text)
        elif isinstance(obj,(list,tuple,set)): #se trova lista, tupla o set, entra dentro ogni elemento
            for item in obj:
                if len(parts)>=max_parts:
                    break
                add_part(item)
        else: #valore semplice, lo converte in testo e lo aggiunge
            text=str(obj).strip()
            if text:
                parts.append(text)

    add_part(parsed)
    return " ".join(parts) #unisce i pezzi in una stringa


def clean_text(text): #pulisce testo finale prima di darlo al TF-IDF
    text=str(text)
    text=re.sub(r"<[^>]+>"," ",text) #rimuove tag HTML
    text=re.sub(r"[^A-Za-z0-9]+"," ",text) #elimina simboli, punteggiatura, parentesi, trattini
    text=re.sub(r"\s+"," ",text) #toglie spazi multipli, e mette spazi singoli
    return text.strip().lower() 


def first_existing_value(row,columns): #prende una riga del Dataframe e una lista di colonne , poi restituisce prima valore non mancante
    for col in columns:
        if col in row.index and not is_missing_value(row[col]):
            value=str(row[col]).strip()
            if value:
                return value
    return ""

In [12]:
def build_product_text(row): #prende riga di items_df, cioè un prodotto e costruisce una stringa testuale pulita
    title=first_existing_value(row,["title"])
    category=first_existing_value(row,["category","main_category"])
    store=first_existing_value(row,["store","brand"])
    features=flatten_text(row["features_json"],max_parts=30) if "features_json" in row.index else flatten_text(row.get("features",[]),max_parts=30)
    description=flatten_text(row["description_json"],max_parts=20) if "description_json" in row.index else flatten_text(row.get("description",[]),max_parts=20)
    categories=flatten_text(row["categories_json"],max_parts=20) if "categories_json" in row.index else flatten_text(row.get("categories",[]),max_parts=20)
    details=flatten_text(row["details_json"],max_parts=12) if "details_json" in row.index else flatten_text(row.get("details",{}),max_parts=12)
    text=" ".join([title,title,category,store,features,description,categories,details]) #unisce tutto in una sola stringa
    return clean_text(text) #lo pulisce e lo ritorna

items_text_df=items_df[[col for col in items_df.columns if col in { #creiamo dataframe più piccolo
    "item_idx","parent_asin","title","category","main_category","store","brand",
    "features_json","description_json","categories_json","details_json",
    "features","description","categories","details","avg_rating","average_rating",
    "num_ratings","rating_number","price","image_url"
}]].copy()

items_text_df["content_text"]=items_text_df.apply(build_product_text,axis=1)  #applica la funzione di sopra, a ogni prodotto
items_text_df=items_text_df.sort_values("item_idx").drop_duplicates("item_idx").reset_index(drop=True) #rimuove duplicati

n_users=int(max(user_mapping_df["user_idx"].max(),train_df["user_idx"].max())+1) #n utenti
n_items=int(max(item_mapping_df["item_idx"].max(),train_df["item_idx"].max(),items_text_df["item_idx"].max())+1) #n items

content_texts=[""]*n_items #lista lunga quanto numero tot di prodotti
for row in items_text_df[["item_idx","content_text"]].itertuples(index=False): #per ogni prodotto,mette il testo nella pos corrispondente al suo item_idx
    content_texts[int(row.item_idx)]=row.content_text

empty_text_count=sum(1 for text in content_texts if not text) #conta testi vuoti
#calcoliamo quante parole ha il testo prodotto, calcoliamo media e percentili
text_length_stats=pd.Series([len(text.split()) for text in content_texts],name="tokens_per_item").describe(percentiles=[0.25,0.5,0.75,0.9,0.95])

print("n_users:",n_users)
print("n_items:",n_items)
print("prodotti con testo vuoto:",empty_text_count)
display(text_length_stats.to_frame())
display(items_text_df[["item_idx","parent_asin","title","content_text"]].head(5))

n_users: 131398
n_items: 12306
prodotti con testo vuoto: 0


,tokens_per_item
count,"12,306.000000"
mean,352.240452
std,255.119731
min,23.000000
25%,184.000000
50%,288.000000
75%,454.000000
90%,649.000000
95%,787.000000
max,"5,974.000000"


,item_idx,parent_asin,title,content_text
0,0,0972683275,"VideoSecu 24"" Long Arm TV Wall Mount Low Profile Articulating Full Motion Swing Tilt Wall Bracket for Most 22"" to 55"" LED LCD UHD TV Monitor Flat Screen VES...",videosecu 24 long arm tv wall mount low profile articulating full motion swing tilt wall bracket for most 22 to 55 led lcd uhd tv monitor flat screen vesa 2...
1,1,1400501466,"Barnes & Noble NOOK Tablet 16gb (Color, BNTV250)",barnes noble nook tablet 16gb color bntv250 barnes noble nook tablet 16gb color bntv250 computers barnes noble nook color tablettm nov 2011 bntv250 hd movie...
2,2,1400532655,"Barnes & Noble NOOK Color 7"" Tablet",barnes noble nook color 7 tablet barnes noble nook color 7 tablet all electronics barnes noble get social surf the web play games even listen to music over ...
3,3,1400532736,NOOK Simple Touch,nook simple touch nook simple touch all electronics barnes noble 6 inch touch screen quick downloads with wi fi easy navigation share reading with friends m...
4,4,1400599997,Barnes & Noble NOOK ebook reader (WiFi + 3G)[B&W],barnes noble nook ebook reader wifi 3g b w barnes noble nook ebook reader wifi 3g b w all electronics barnes noble barnes noble brand ebook reader nook e in...
